In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))

In [2]:
from modules.awa.awa_block import AnchorWindowAttention

In [3]:
import torch

In [4]:
BATCH = 2
SEQ_LEN = 4096
D_MODEL = 128
NUM_HEADS = 8
HEAD_DIM = D_MODEL // NUM_HEADS
DTYPE = torch.float16
DEVICE = 'cuda'

WINDOW_SIZE = 128
NUM_META_TOKENS = 4

model = AnchorWindowAttention(
    embed_dim=HEAD_DIM, 
    window_size=WINDOW_SIZE, 
    meta_token=NUM_META_TOKENS,
    dtype=DTYPE
)
model = model.to(device=DEVICE)

q = torch.randn(BATCH, NUM_HEADS, SEQ_LEN, HEAD_DIM, device=DEVICE, dtype=DTYPE)
k = torch.randn(BATCH, NUM_HEADS, SEQ_LEN, HEAD_DIM, device=DEVICE, dtype=DTYPE)
v = torch.randn(BATCH, NUM_HEADS, SEQ_LEN, HEAD_DIM, device=DEVICE, dtype=DTYPE)

print(f"Benchmarking AnchorWindowAttention...")
print(f"Config: B={BATCH}, L={SEQ_LEN}, H={NUM_HEADS}, D={HEAD_DIM}")
print(f"Window Size: {WINDOW_SIZE}, Anchors: {NUM_META_TOKENS}")

print("\nWarming up GPU...")
for _ in range(10):
    _ = model(q, k, v)
torch.cuda.synchronize()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

start_event.record()

n_loops = 100
for _ in range(n_loops):
    y = model(q, k, v)

end_event.record()
torch.cuda.synchronize()

elapsed_time_ms = start_event.elapsed_time(end_event)
avg_time = elapsed_time_ms / n_loops

print(f"Output shape: {y.shape}")
print(f"Total time for {n_loops} runs: {elapsed_time_ms:.2f} ms")
print(f"Average time per forward pass: {avg_time:.4f} ms")

Benchmarking AnchorWindowAttention...
Config: B=2, L=4096, H=8, D=16
Window Size: 128, Anchors: 4

Warming up GPU...
Output shape: torch.Size([2, 8, 4096, 16])
Total time for 100 runs: 539.02 ms
Average time per forward pass: 5.3902 ms
